# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es entrenar un modelo de Random Forest sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~0.66 F1-macro en Kaggle).

Random Forest es un ensemble de árboles de decisión entrenados sobre subconjuntos aleatorios del data y de las features. Al promediar muchos árboles distintos reduce la varianza y generaliza mejor que un árbol solo.

## Importación e instalación de dependencias


In [1]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 94.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

In [3]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [4]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [5]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N4: Bayes Naive + TF-IDF con RandomizedSearch

Búsqueda de hiperparámetros sobre RF + TF-IDF con 40 iteraciones. Se usa RandomizedSearchCV en lugar de GridSearchCV porque el espacio de hiperparámetros de RF es grande y GridSearch tardaría demasiado.

Hiperparámetros explorados:
- tfidf__ngram_range: unigramas vs unigramas+bigramas
- tfidf__min_df: mínima frecuencia de documento
- rf__n_estimators: cantidad de árboles del ensemble
- rf__max_depth: profundidad máxima de cada árbol (None = sin límite)
- rf__min_samples_leaf: mínimo de muestras en una hoja del árbol
- rf__max_features: features consideradas en cada split (sqrt o log2 del total)

In [6]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("rf",    RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1))])

param_rs_tfidf = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__sublinear_tf": [True, False],
    "rf__n_estimators": [100, 200, 300, 400],
    "rf__max_depth": [None, 50, 80, 100],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2"],
}

rs_rf_tfidf = RandomizedSearchCV(
    pipe,
    param_rs_tfidf,
    n_iter=40,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=SEED,
    verbose=1
)

print("Buscando mejores hiperparámetros RF + TF-IDF")
rs_rf_tfidf.fit(X_train, y_train)

print(f"\nMejores hiperparámetros: {rs_rf_tfidf.best_params_}")
print(f"Mejor F1-macro en CV: {rs_rf_tfidf.best_score_:.4f}")

y_pred = rs_rf_tfidf.best_estimator_.predict(X_val)
evaluate("rf_tfidf_randomsearch_v4", y_val, y_pred, hyperparams=rs_rf_tfidf.best_params_)

# Reentrenar en train completo y generar submission
best_pipe = rs_rf_tfidf.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(best_pipe, "rf_tfidf_randomsearch_v4")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_rf_tfidf_randomsearch_v4.csv");

Buscando mejores hiperparámetros RF + TF-IDF
Fitting 5 folds for each of 40 candidates, totalling 200 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Mejores hiperparámetros: {'tfidf__sublinear_tf': False, 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'rf__n_estimators': 300, 'rf__min_samples_leaf': 2, 'rf__max_features': 'sqrt', 'rf__max_depth': 100}
Mejor F1-macro en CV: 0.6369

=== rf_tfidf_randomsearch_v4 ===
Hiperparámetros: {'tfidf__sublinear_tf': False, 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'rf__n_estimators': 300, 'rf__min_samples_leaf': 2, 'rf__max_features': 'sqrt', 'rf__max_depth': 100}

F1-macro:  0.6418
Precision: 0.6420
Recall:    0.6416
Accuracy:  0.6898

              precision    recall  f1-score   support

    negativa     0.7439    0.7475    0.7457      4080
      neutra     0.4164    0.4005    0.4083      2040
    positiva     0.7658    0.7767    0.7712      4080

    accuracy                         0.6898     10200
   macro avg     0.6420    0.6416    0.6418     10200
weighted avg     0.6872    0.6898    0.6884     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neut